In [ ]:
import os
import pandas as pd

os.makedirs("../outputs_dedup_correction", exist_ok=True)

df_train_dedup = pd.read_csv("../data_dedup/train.csv")
df_test_dedup = pd.read_csv("../data_dedup/test.csv")

print("✅ Dossier créé et dataset (dédupliqué) rechargé !")
print(f"Train : {df_train_dedup.shape}")
print(f"Test  : {df_test_dedup.shape}")

# Vérifier que Plain_Text est bien sans emojis
exemple = df_train_dedup["Plain_Text"].iloc[4]
print(f"\nPlain_Text (exemple) : {exemple}")

import emoji
contient_emoji = any(c in emoji.EMOJI_DATA for c in exemple)
print(f"Contient des emojis : {contient_emoji}")

In [ ]:
exemple = df_train_dedup["Plain_Text"].iloc[4]
print(f"Plain_Text : {exemple}")

import emoji
contient_emoji = any(c in emoji.EMOJI_DATA for c in exemple)
print(f"\nContient des emojis : {contient_emoji}")

In [ ]:
#charger marbert

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModel

tokenizer = AutoTokenizer.from_pretrained("UBC-NLP/MARBERT")
model = AutoModel.from_pretrained("UBC-NLP/MARBERT")
model.eval()

print("✅ MARBERT chargé !")

longueurs = df_train_dedup["Plain_Text"].apply(
    lambda t: len(tokenizer.tokenize(t)) if isinstance(t, str) else 0
)
print(f"\nLongueur moyenne : {longueurs.mean():.1f} tokens")
print(f"95e percentile   : {longueurs.quantile(0.95):.1f} tokens")
print(f"99e percentile   : {longueurs.quantile(0.99):.1f} tokens")

In [ ]:
#extraction complète des embeddings MARBERT sur Plain_Text

In [ ]:
import numpy as np
from tqdm import tqdm

MAX_LENGTH = 64

def get_embedding_sequence(text):
    if not isinstance(text, str):
        text = ""
    inputs = tokenizer(text, return_tensors="pt",
                       truncation=True, max_length=MAX_LENGTH,
                       padding="max_length")
    with torch.no_grad():
        outputs = model(**inputs)
    sequence = outputs.last_hidden_state.squeeze(0).numpy()
    return sequence

N_train = len(df_train_dedup)
N_test = len(df_test_dedup)
seq_shape = (MAX_LENGTH, 768)

print(f"⏳ Extraction + sauvegarde directe train ({N_train} tweets)...")

train_mmap = np.lib.format.open_memmap(
    "../outputs_dedup_correction/train_embeddings_sequence.npy",
    mode='w+', dtype=np.float32, shape=(N_train,) + seq_shape
)

textes_train = df_train_dedup["Plain_Text"].tolist()
for i in tqdm(range(N_train)):
    train_mmap[i] = get_embedding_sequence(textes_train[i])

train_mmap.flush()
del train_mmap

print("✅ Train embeddings sauvegardés !")

In [ ]:
print(f"⏳ Extraction + sauvegarde directe test ({N_test} tweets)...")

test_mmap = np.lib.format.open_memmap(
    "../outputs_dedup_correction/test_embeddings_sequence.npy",
    mode='w+', dtype=np.float32, shape=(N_test,) + seq_shape
)

textes_test = df_test_dedup["Plain_Text"].tolist()
for i in tqdm(range(N_test)):
    test_mmap[i] = get_embedding_sequence(textes_test[i])

test_mmap.flush()
del test_mmap

print("✅ Test embeddings sauvegardés !")

np.save("../outputs_dedup_correction/train_labels.npy", df_train_dedup["label"].astype(int).values)
np.save("../outputs_dedup_correction/test_labels.npy", df_test_dedup["label"].astype(int).values)

print("✅ Tous les fichiers sauvegardés dans outputs_dedup_correction/ !")

In [ ]:
import gc
import pandas as pd
from sklearn.model_selection import train_test_split

gc.collect()

X_train_seq_correction = np.load("../outputs_dedup_correction/train_embeddings_sequence.npy", mmap_mode='r')
X_test_seq_correction = np.load("../outputs_dedup_correction/test_embeddings_sequence.npy", mmap_mode='r')

print("✅ Séquences MARBERT (Plain_Text, corrigées) rechargées !")
print(f"Train : {X_train_seq_correction.shape}")
print(f"Test  : {X_test_seq_correction.shape}")

train_prag_dedup = pd.read_csv("../outputs_dedup/train_pragmatic_v2.csv")
test_prag_dedup = pd.read_csv("../outputs_dedup/test_pragmatic_v2.csv")

X_train_prag_correction = train_prag_dedup.drop("label", axis=1).values
X_test_prag_correction = test_prag_dedup.drop("label", axis=1).values

y_train_full_correction = np.load("../outputs_dedup_correction/train_labels.npy")
y_test_correction = np.load("../outputs_dedup_correction/test_labels.npy")

indices_correction = np.arange(len(y_train_full_correction))
idx_train_correction, idx_val_correction = train_test_split(
    indices_correction, test_size=0.15, stratify=y_train_full_correction, random_state=42
)

print(f"\n✅ Split train/val créé !")
print(f"Train final : {len(idx_train_correction)} | Val : {len(idx_val_correction)}")

In [ ]:
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import copy
from sklearn.metrics import f1_score, matthews_corrcoef, classification_report, precision_score, recall_score, accuracy_score
from scipy.stats import wilcoxon

class SoftAttention(nn.Module):
    def __init__(self, hidden_dim):
        super(SoftAttention, self).__init__()
        self.attention = nn.Linear(hidden_dim, 1)
    def forward(self, lstm_output):
        scores = self.attention(lstm_output).squeeze(-1)
        weights = F.softmax(scores, dim=1)
        context = torch.bmm(weights.unsqueeze(1), lstm_output).squeeze(1)
        return context, weights

class SAttBLSTMConvNet(nn.Module):
    def __init__(self, embed_dim=768, conv_channels=128, lstm_hidden=128, prag_dim=18, dropout=0.3):
        super(SAttBLSTMConvNet, self).__init__()
        self.conv1 = nn.Conv1d(embed_dim, conv_channels, kernel_size=3, padding=1)
        self.relu = nn.ReLU()
        self.dropout_conv = nn.Dropout(dropout)
        self.bilstm = nn.LSTM(conv_channels, lstm_hidden, batch_first=True, bidirectional=True)
        self.attention = SoftAttention(lstm_hidden * 2)
        self.dropout_final = nn.Dropout(dropout)
        self.fc1 = nn.Linear(lstm_hidden * 2 + prag_dim, 128)
        self.fc2 = nn.Linear(128, 2)
    def forward(self, x_seq, x_prag):
        x = x_seq.permute(0, 2, 1)
        x = self.relu(self.conv1(x))
        x = self.dropout_conv(x)
        x = x.permute(0, 2, 1)
        lstm_out, _ = self.bilstm(x)
        context, attn_weights = self.attention(lstm_out)
        combined = torch.cat([context, x_prag], dim=1)
        combined = self.dropout_final(combined)
        out = self.relu(self.fc1(combined))
        return self.fc2(out)

class SarcasmDatasetMemmap(Dataset):
    def __init__(self, X_seq_memmap, X_prag, y, indices):
        self.X_seq_memmap = X_seq_memmap
        self.X_prag = torch.FloatTensor(X_prag[indices])
        self.y = torch.LongTensor(y[indices])
        self.indices = indices
    def __len__(self):
        return len(self.indices)
    def __getitem__(self, idx):
        seq = np.array(self.X_seq_memmap[self.indices[idx]])
        return torch.FloatTensor(seq), self.X_prag[idx], self.y[idx]

def entrainer_avec_early_stopping_memmap(X_seq_memmap, X_prag, y, idx_train, idx_val,
                                           lr=0.001, max_epochs=50, patience=10, batch_size=32):
    train_dataset = SarcasmDatasetMemmap(X_seq_memmap, X_prag, y, idx_train)
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
    val_dataset = SarcasmDatasetMemmap(X_seq_memmap, X_prag, y, idx_val)
    val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)
    class_counts = np.bincount(y[idx_train])
    weights = torch.FloatTensor(1.0 / class_counts)
    model = SAttBLSTMConvNet(embed_dim=768, prag_dim=X_prag.shape[1])
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    criterion = nn.CrossEntropyLoss(weight=weights)
    best_f1 = 0
    best_model_state = None
    epochs_sans_amelioration = 0
    for epoch in range(max_epochs):
        model.train()
        total_loss = 0
        for x_seq_b, x_prag_b, y_b in train_loader:
            optimizer.zero_grad()
            outputs = model(x_seq_b, x_prag_b)
            loss = criterion(outputs, y_b)
            loss.backward()
            optimizer.step()
            total_loss += loss.item()
        model.eval()
        all_preds, all_labels = [], []
        with torch.no_grad():
            for x_seq_b, x_prag_b, y_b in val_loader:
                outputs = model(x_seq_b, x_prag_b)
                _, preds = torch.max(outputs, 1)
                all_preds.extend(preds.numpy())
                all_labels.extend(y_b.numpy())
        f1_val = f1_score(all_labels, all_preds, average='macro')
        if (epoch + 1) % 5 == 0 or epoch == 0:
            print(f"   Epoch {epoch+1}/{max_epochs} - Loss: {total_loss/len(train_loader):.4f} - F1 val: {f1_val:.4f}")
        if f1_val > best_f1:
            best_f1 = f1_val
            best_model_state = copy.deepcopy(model.state_dict())
            epochs_sans_amelioration = 0
        else:
            epochs_sans_amelioration += 1
        if epochs_sans_amelioration >= patience:
            print(f"   ⏹️ Early stopping à l'époque {epoch+1}")
            break
    model.load_state_dict(best_model_state)
    print(f"✅ Meilleur F1 validation atteint : {best_f1:.4f}")
    return model

def evaluer_test_complet_memmap(model, X_seq_memmap, X_prag, y, idx_test, batch_size=32):
    test_dataset = SarcasmDatasetMemmap(X_seq_memmap, X_prag, y, idx_test)
    test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)
    model.eval()
    all_preds, all_labels = [], []
    with torch.no_grad():
        for x_seq_b, x_prag_b, y_b in test_loader:
            outputs = model(x_seq_b, x_prag_b)
            _, preds = torch.max(outputs, 1)
            all_preds.extend(preds.numpy())
            all_labels.extend(y_b.numpy())
    resultats = {
        "F1_macro": f1_score(all_labels, all_preds, average='macro'),
        "Precision_macro": precision_score(all_labels, all_preds, average='macro'),
        "Recall_macro": recall_score(all_labels, all_preds, average='macro'),
        "Accuracy": accuracy_score(all_labels, all_preds),
        "MCC": matthews_corrcoef(all_labels, all_preds),
        "F1_sarcastique": f1_score(all_labels, all_preds, pos_label=1),
        "Precision_sarcastique": precision_score(all_labels, all_preds, pos_label=1),
        "Recall_sarcastique": recall_score(all_labels, all_preds, pos_label=1)
    }
    print(classification_report(all_labels, all_preds))
    print(f"\n{'='*50}\nRÉSUMÉ DES MÉTRIQUES\n{'='*50}")
    for nom, valeur in resultats.items():
        print(f"{nom:25} : {valeur:.4f}")
    return resultats

print("✅ Architecture et fonctions prêtes !")

In [ ]:
def evaluer_hyperparam_correction(lr, epochs=5, batch_size=32):
    torch.manual_seed(42)
    train_dataset = SarcasmDatasetMemmap(X_train_seq_correction, X_train_prag_correction, y_train_full_correction, idx_train_correction)
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
    val_dataset = SarcasmDatasetMemmap(X_train_seq_correction, X_train_prag_correction, y_train_full_correction, idx_val_correction)
    val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)
    class_counts = np.bincount(y_train_full_correction[idx_train_correction])
    weights = torch.FloatTensor(1.0 / class_counts)
    model = SAttBLSTMConvNet(embed_dim=768, prag_dim=18)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    criterion = nn.CrossEntropyLoss(weight=weights)
    model.train()
    for epoch in range(epochs):
        for x_seq, x_prag, y in train_loader:
            optimizer.zero_grad()
            outputs = model(x_seq, x_prag)
            loss = criterion(outputs, y)
            loss.backward()
            optimizer.step()
    model.eval()
    all_preds, all_labels = [], []
    with torch.no_grad():
        for x_seq, x_prag, y in val_loader:
            outputs = model(x_seq, x_prag)
            _, preds = torch.max(outputs, 1)
            all_preds.extend(preds.numpy())
            all_labels.extend(y.numpy())
    return f1_score(all_labels, all_preds, average='macro')

print("⏳ Test rapide (MARBERT corrigé, Plain_Text)...")
f1_test = evaluer_hyperparam_correction(0.001, epochs=5)
print(f"F1 macro (validation, 5 époques) = {f1_test:.4f}")

In [ ]:
seeds = [42, 123, 456, 789, 101]

resultats_5runs_sans_correction = []

X_train_prag_zeros_correction = np.zeros_like(X_train_prag_correction)
X_test_prag_zeros_correction = np.zeros_like(X_test_prag_correction)

print("="*60)
print("5 RUNS — SANS features (ArSarcaMoji, MARBERT CORRIGÉ)")
print("="*60)

for i, seed in enumerate(seeds):
    print(f"\n--- Run {i+1}/5 (seed={seed}) ---")
    torch.manual_seed(seed)
    np.random.seed(seed)
    
    model_s = entrainer_avec_early_stopping_memmap(
        X_train_seq_correction, X_train_prag_zeros_correction, y_train_full_correction,
        idx_train_correction, idx_val_correction,
        lr=0.001, max_epochs=50, patience=10
    )
    
    idx_test_all_correction = np.arange(len(y_test_correction))
    resultats_metriques = evaluer_test_complet_memmap(
        model_s, X_test_seq_correction, X_test_prag_zeros_correction, y_test_correction, idx_test_all_correction
    )
    resultats_metriques["seed"] = seed
    resultats_5runs_sans_correction.append(resultats_metriques)

print("\n✅ 5 runs SANS features terminés !")
pd.DataFrame(resultats_5runs_sans_correction).to_csv("../outputs_dedup_correction/5runs_sans_features_complet.csv", index=False)
print("✅ Résultats sauvegardés !")

In [ ]:
seeds = [42, 123, 456, 789, 101]

resultats_5runs_sans_correction = []

X_train_prag_zeros_correction = np.zeros_like(X_train_prag_correction)
X_test_prag_zeros_correction = np.zeros_like(X_test_prag_correction)

print("="*60)
print("5 RUNS — SANS features (ArSarcaMoji, MARBERT CORRIGÉ)")
print("="*60)

for i, seed in enumerate(seeds):
    print(f"\n--- Run {i+1}/5 (seed={seed}) ---")
    torch.manual_seed(seed)
    np.random.seed(seed)
    
    model_s = entrainer_avec_early_stopping_memmap(
        X_train_seq_correction, X_train_prag_zeros_correction, y_train_full_correction,
        idx_train_correction, idx_val_correction,
        lr=0.001, max_epochs=50, patience=10
    )
    
    idx_test_all_correction = np.arange(len(y_test_correction))
    resultats_metriques = evaluer_test_complet_memmap(
        model_s, X_test_seq_correction, X_test_prag_zeros_correction, y_test_correction, idx_test_all_correction
    )
    resultats_metriques["seed"] = seed
    resultats_5runs_sans_correction.append(resultats_metriques)

print("\n✅ 5 runs SANS features terminés !")
pd.DataFrame(resultats_5runs_sans_correction).to_csv("../outputs_dedup_correction/5runs_sans_features_complet.csv", index=False)
print("✅ Résultats sauvegardés !")

In [ ]:
resultats_5runs_avec_correction = []

print("="*60)
print("5 RUNS — AVEC features (ArSarcaMoji, MARBERT CORRIGÉ)")
print("="*60)

for i, seed in enumerate(seeds):
    print(f"\n--- Run {i+1}/5 (seed={seed}) ---")
    torch.manual_seed(seed)
    np.random.seed(seed)
    
    model_a = entrainer_avec_early_stopping_memmap(
        X_train_seq_correction, X_train_prag_correction, y_train_full_correction,
        idx_train_correction, idx_val_correction,
        lr=0.001, max_epochs=50, patience=10
    )
    
    resultats_metriques = evaluer_test_complet_memmap(
        model_a, X_test_seq_correction, X_test_prag_correction, y_test_correction, idx_test_all_correction
    )
    resultats_metriques["seed"] = seed
    resultats_5runs_avec_correction.append(resultats_metriques)

print("\n✅ 5 runs AVEC features terminés !")
pd.DataFrame(resultats_5runs_avec_correction).to_csv("../outputs_dedup_correction/5runs_avec_features_complet.csv", index=False)
print("✅ Résultats sauvegardés !")

In [ ]:
f1_sans_correction = [r["F1_macro"] for r in resultats_5runs_sans_correction]
f1_avec_correction = [r["F1_macro"] for r in resultats_5runs_avec_correction]

print("F1 SANS :", [f"{f:.4f}" for f in f1_sans_correction])
print("F1 AVEC :", [f"{f:.4f}" for f in f1_avec_correction])

print(f"\nSANS : {np.mean(f1_sans_correction):.4f} ± {np.std(f1_sans_correction):.4f}")
print(f"AVEC : {np.mean(f1_avec_correction):.4f} ± {np.std(f1_avec_correction):.4f}")

stat, p_value = wilcoxon(f1_avec_correction, f1_sans_correction, alternative='greater')
print(f"\nWilcoxon (ArSarcaMoji, MARBERT CORRIGÉ) : p = {p_value:.4f}")

if p_value < 0.05:
    print("✅✅ SIGNIFICATIF !")


In [ ]:
f1_sans_correction = [r["F1_macro"] for r in resultats_5runs_sans_correction]
f1_avec_correction = [r["F1_macro"] for r in resultats_5runs_avec_correction]

print("F1 SANS :", [f"{f:.4f}" for f in f1_sans_correction])
print("F1 AVEC :", [f"{f:.4f}" for f in f1_avec_correction])

print(f"\nSANS : {np.mean(f1_sans_correction):.4f} ± {np.std(f1_sans_correction):.4f}")
print(f"AVEC : {np.mean(f1_avec_correction):.4f} ± {np.std(f1_avec_correction):.4f}")

stat, p_value = wilcoxon(f1_avec_correction, f1_sans_correction, alternative='greater')
print(f"\nWilcoxon (ArSarcaMoji, MARBERT CORRIGÉ) : p = {p_value:.4f}")

if p_value < 0.05:
    print("✅✅ SIGNIFICATIF !")

In [ ]:
# Recréer manuellement les résultats SANS features (déjà obtenus précédemment)
f1_sans_correction = [0.6873, 0.6957, 0.6931, 0.6986, 0.7075]
f1_avec_correction = [r["F1_macro"] for r in resultats_5runs_avec_correction]

print("F1 SANS :", [f"{f:.4f}" for f in f1_sans_correction])
print("F1 AVEC :", [f"{f:.4f}" for f in f1_avec_correction])

print(f"\nSANS : {np.mean(f1_sans_correction):.4f} ± {np.std(f1_sans_correction):.4f}")
print(f"AVEC : {np.mean(f1_avec_correction):.4f} ± {np.std(f1_avec_correction):.4f}")

stat, p_value = wilcoxon(f1_avec_correction, f1_sans_correction, alternative='greater')
print(f"\nWilcoxon (ArSarcaMoji, MARBERT CORRIGÉ) : p = {p_value:.4f}")

if p_value < 0.05:
    print("✅✅ SIGNIFICATIF !")
elif p_value < 0.10:
    print("⚠️ Marginalement significatif")
else:
    print("⚠️ Non significatif")

In [ ]:
#ablation

In [ ]:
torch.manual_seed(42)
np.random.seed(42)

model_ablation_correction = entrainer_avec_early_stopping_memmap(
    X_train_seq_correction, X_train_prag_correction, y_train_full_correction,
    idx_train_correction, idx_val_correction,
    lr=0.001, max_epochs=50, patience=10
)

def evaluer_fige_features_modifiees(model, X_seq_memmap, X_prag_modif, y, idx):
    dataset = SarcasmDatasetMemmap(X_seq_memmap, X_prag_modif, y, idx)
    loader = DataLoader(dataset, batch_size=32, shuffle=False)
    model.eval()
    all_preds, all_labels = [], []
    with torch.no_grad():
        for x_seq_b, x_prag_b, y_b in loader:
            outputs = model(x_seq_b, x_prag_b)
            _, preds = torch.max(outputs, 1)
            all_preds.extend(preds.numpy())
            all_labels.extend(y_b.numpy())
    return f1_score(all_labels, all_preds, average='macro')

f1_base_ablation_correction = evaluer_fige_features_modifiees(
    model_ablation_correction, X_test_seq_correction, X_test_prag_correction,
    y_test_correction, idx_test_all_correction
)
print(f"\n✅ F1 de référence pour l'ablation (MARBERT corrigé) : {f1_base_ablation_correction:.4f}")

In [ ]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import copy
from sklearn.metrics import f1_score, matthews_corrcoef, classification_report, precision_score, recall_score, accuracy_score
from scipy.stats import wilcoxon
from sklearn.model_selection import train_test_split
import gc

print("✅ Imports de base chargés !")

In [ ]:
gc.collect()

X_train_seq_correction = np.load("../outputs_dedup_correction/train_embeddings_sequence.npy", mmap_mode='r')
X_test_seq_correction = np.load("../outputs_dedup_correction/test_embeddings_sequence.npy", mmap_mode='r')

train_prag_dedup = pd.read_csv("../outputs_dedup/train_pragmatic_v2.csv")
test_prag_dedup = pd.read_csv("../outputs_dedup/test_pragmatic_v2.csv")

X_train_prag_correction = train_prag_dedup.drop("label", axis=1).values
X_test_prag_correction = test_prag_dedup.drop("label", axis=1).values

y_train_full_correction = np.load("../outputs_dedup_correction/train_labels.npy")
y_test_correction = np.load("../outputs_dedup_correction/test_labels.npy")

indices_correction = np.arange(len(y_train_full_correction))
idx_train_correction, idx_val_correction = train_test_split(
    indices_correction, test_size=0.15, stratify=y_train_full_correction, random_state=42
)
idx_test_all_correction = np.arange(len(y_test_correction))

print("✅ Données MARBERT corrigées rechargées !")
print(f"Train seq : {X_train_seq_correction.shape}")
print(f"Test seq  : {X_test_seq_correction.shape}")

In [ ]:
class SoftAttention(nn.Module):
    def __init__(self, hidden_dim):
        super(SoftAttention, self).__init__()
        self.attention = nn.Linear(hidden_dim, 1)
    def forward(self, lstm_output):
        scores = self.attention(lstm_output).squeeze(-1)
        weights = F.softmax(scores, dim=1)
        context = torch.bmm(weights.unsqueeze(1), lstm_output).squeeze(1)
        return context, weights

class SAttBLSTMConvNet(nn.Module):
    def __init__(self, embed_dim=768, conv_channels=128, lstm_hidden=128, prag_dim=18, dropout=0.3):
        super(SAttBLSTMConvNet, self).__init__()
        self.conv1 = nn.Conv1d(embed_dim, conv_channels, kernel_size=3, padding=1)
        self.relu = nn.ReLU()
        self.dropout_conv = nn.Dropout(dropout)
        self.bilstm = nn.LSTM(conv_channels, lstm_hidden, batch_first=True, bidirectional=True)
        self.attention = SoftAttention(lstm_hidden * 2)
        self.dropout_final = nn.Dropout(dropout)
        self.fc1 = nn.Linear(lstm_hidden * 2 + prag_dim, 128)
        self.fc2 = nn.Linear(128, 2)
    def forward(self, x_seq, x_prag):
        x = x_seq.permute(0, 2, 1)
        x = self.relu(self.conv1(x))
        x = self.dropout_conv(x)
        x = x.permute(0, 2, 1)
        lstm_out, _ = self.bilstm(x)
        context, attn_weights = self.attention(lstm_out)
        combined = torch.cat([context, x_prag], dim=1)
        combined = self.dropout_final(combined)
        out = self.relu(self.fc1(combined))
        return self.fc2(out)

class SarcasmDatasetMemmap(Dataset):
    def __init__(self, X_seq_memmap, X_prag, y, indices):
        self.X_seq_memmap = X_seq_memmap
        self.X_prag = torch.FloatTensor(X_prag[indices])
        self.y = torch.LongTensor(y[indices])
        self.indices = indices
    def __len__(self):
        return len(self.indices)
    def __getitem__(self, idx):
        seq = np.array(self.X_seq_memmap[self.indices[idx]])
        return torch.FloatTensor(seq), self.X_prag[idx], self.y[idx]

def entrainer_avec_early_stopping_memmap(X_seq_memmap, X_prag, y, idx_train, idx_val,
                                           lr=0.001, max_epochs=50, patience=10, batch_size=32):
    train_dataset = SarcasmDatasetMemmap(X_seq_memmap, X_prag, y, idx_train)
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
    val_dataset = SarcasmDatasetMemmap(X_seq_memmap, X_prag, y, idx_val)
    val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)
    class_counts = np.bincount(y[idx_train])
    weights = torch.FloatTensor(1.0 / class_counts)
    model = SAttBLSTMConvNet(embed_dim=768, prag_dim=X_prag.shape[1])
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    criterion = nn.CrossEntropyLoss(weight=weights)
    best_f1 = 0
    best_model_state = None
    epochs_sans_amelioration = 0
    for epoch in range(max_epochs):
        model.train()
        total_loss = 0
        for x_seq_b, x_prag_b, y_b in train_loader:
            optimizer.zero_grad()
            outputs = model(x_seq_b, x_prag_b)
            loss = criterion(outputs, y_b)
            loss.backward()
            optimizer.step()
            total_loss += loss.item()
        model.eval()
        all_preds, all_labels = [], []
        with torch.no_grad():
            for x_seq_b, x_prag_b, y_b in val_loader:
                outputs = model(x_seq_b, x_prag_b)
                _, preds = torch.max(outputs, 1)
                all_preds.extend(preds.numpy())
                all_labels.extend(y_b.numpy())
        f1_val = f1_score(all_labels, all_preds, average='macro')
        if (epoch + 1) % 5 == 0 or epoch == 0:
            print(f"   Epoch {epoch+1}/{max_epochs} - Loss: {total_loss/len(train_loader):.4f} - F1 val: {f1_val:.4f}")
        if f1_val > best_f1:
            best_f1 = f1_val
            best_model_state = copy.deepcopy(model.state_dict())
            epochs_sans_amelioration = 0
        else:
            epochs_sans_amelioration += 1
        if epochs_sans_amelioration >= patience:
            print(f"   ⏹️ Early stopping à l'époque {epoch+1}")
            break
    model.load_state_dict(best_model_state)
    print(f"✅ Meilleur F1 validation atteint : {best_f1:.4f}")
    return model

print("✅ Architecture et fonctions redéfinies !")

In [ ]:
torch.manual_seed(42)
np.random.seed(42)

model_ablation_correction = entrainer_avec_early_stopping_memmap(
    X_train_seq_correction, X_train_prag_correction, y_train_full_correction,
    idx_train_correction, idx_val_correction,
    lr=0.001, max_epochs=50, patience=10
)

def evaluer_fige_features_modifiees(model, X_seq_memmap, X_prag_modif, y, idx):
    dataset = SarcasmDatasetMemmap(X_seq_memmap, X_prag_modif, y, idx)
    loader = DataLoader(dataset, batch_size=32, shuffle=False)
    model.eval()
    all_preds, all_labels = [], []
    with torch.no_grad():
        for x_seq_b, x_prag_b, y_b in loader:
            outputs = model(x_seq_b, x_prag_b)
            _, preds = torch.max(outputs, 1)
            all_preds.extend(preds.numpy())
            all_labels.extend(y_b.numpy())
    return f1_score(all_labels, all_preds, average='macro')

f1_base_ablation_correction = evaluer_fige_features_modifiees(
    model_ablation_correction, X_test_seq_correction, X_test_prag_correction,
    y_test_correction, idx_test_all_correction
)
print(f"\n✅ F1 de référence pour l'ablation (MARBERT corrigé) : {f1_base_ablation_correction:.4f}")

In [ ]:
feature_names_v2 = [
    'p_pos', 'p_neg', 'p_neu',
    'intensifier', 'irony', 'neg_pos', 'neg_neg',
    'hyperbole', 'exclamation', 'question', 'latin_ratio',
    'discriminant_vocab', 'hashtag_count', 'emoji_count',
    'interjection', 'quotation', 'repeated_chars', 'tweet_length'
]

print(f"F1 de référence : {f1_base_ablation_correction:.4f}\n")
print("⏳ LOFO en cours (rapide, pas de ré-entraînement)...")

resultats_lofo_correction = []

for i, feature_name in enumerate(feature_names_v2):
    X_test_prag_zero = X_test_prag_correction.copy()
    X_test_prag_zero[:, i] = 0
    
    f1 = evaluer_fige_features_modifiees(
        model_ablation_correction, X_test_seq_correction, X_test_prag_zero,
        y_test_correction, idx_test_all_correction
    )
    impact = f1_base_ablation_correction - f1
    
    print(f"Sans {feature_name:20} → F1 = {f1:.4f} | Impact = {impact:+.4f}")
    resultats_lofo_correction.append({
        "Feature": feature_name,
        "F1 sans feature": round(f1, 4),
        "Impact": round(impact, 4)
    })

lofo_correction_df = pd.DataFrame(resultats_lofo_correction)
lofo_correction_df = lofo_correction_df.sort_values("Impact", ascending=False)

print("\n✅ LOFO terminé !")
print(lofo_correction_df.to_string())
lofo_correction_df.to_csv("../outputs_dedup_correction/lofo_18features_final.csv", index=False)

In [ ]:
idx_sentiment = [0, 1, 2]
idx_lexical = [3, 4, 5, 6, 7, 8, 9, 10]
idx_nouvelles = [11, 12, 13, 14, 15, 16, 17]

print(f"F1 de référence : {f1_base_ablation_correction:.4f}\n")

resultats_group_correction = []

for nom_groupe, indices in [
    ("Sentiment Proxy", idx_sentiment),
    ("Lexical-Pragmatique original", idx_lexical),
    ("Nouvelles features enrichies", idx_nouvelles)
]:
    X_test_prag_zero = X_test_prag_correction.copy()
    X_test_prag_zero[:, indices] = 0
    
    f1 = evaluer_fige_features_modifiees(
        model_ablation_correction, X_test_seq_correction, X_test_prag_zero,
        y_test_correction, idx_test_all_correction
    )
    impact = f1_base_ablation_correction - f1
    print(f"Sans groupe '{nom_groupe}' ({len(indices)} features) :")
    print(f"   F1 = {f1:.4f} | Impact = {impact:+.4f}\n")
    
    resultats_group_correction.append({
        "Groupe": nom_groupe, "Nb_features": len(indices),
        "F1_sans_groupe": round(f1, 4), "Impact": round(impact, 4)
    })

group_correction_df = pd.DataFrame(resultats_group_correction)
group_correction_df.to_csv("../outputs_dedup_correction/group_ablation_18features_final.csv", index=False)
print("✅ Group-Level Ablation terminée !")

In [ ]:
import random
import matplotlib.pyplot as plt

ordre_lofo_correction = lofo_correction_df["Feature"].tolist()
indices_ordre_correction = [feature_names_v2.index(f) for f in ordre_lofo_correction]

def evaluer_cumulative_correction(ordre_indices):
    f1_scores = [f1_base_ablation_correction]
    X_test_prag_modif = X_test_prag_correction.copy()
    for idx in ordre_indices:
        X_test_prag_modif = X_test_prag_modif.copy()
        X_test_prag_modif[:, idx] = 0
        f1 = evaluer_fige_features_modifiees(
            model_ablation_correction, X_test_seq_correction, X_test_prag_modif,
            y_test_correction, idx_test_all_correction
        )
        f1_scores.append(f1)
    return f1_scores

print("⏳ Courbe dans l'ordre d'importance...")
f1_courbe_correction = evaluer_cumulative_correction(indices_ordre_correction)
print("✅ Terminé !")

random.seed(42)
courbes_aleatoires_correction = []
for run in range(5):
    ordre_alea = list(range(18))
    random.shuffle(ordre_alea)
    courbe = evaluer_cumulative_correction(ordre_alea)
    courbes_aleatoires_correction.append(courbe)
    print(f"Run aléatoire {run+1}/5 terminé")

courbes_array = np.array(courbes_aleatoires_correction)
moyenne_alea = courbes_array.mean(axis=0)
std_alea = courbes_array.std(axis=0)
etapes = list(range(19))

fig, ax = plt.subplots(figsize=(12, 6))
ax.fill_between(etapes, moyenne_alea - std_alea, moyenne_alea + std_alea,
                color='grey', alpha=0.3, label='Ordre aleatoire (moyenne +/- std)')
ax.plot(etapes, moyenne_alea, color='grey', linestyle='--', linewidth=1)
ax.plot(etapes, f1_courbe_correction, color='red', marker='o', linewidth=2, label="Ordre d'importance (LOFO)")
ax.axhline(y=f1_base_ablation_correction, color='blue', linestyle=':', linewidth=1.5, label=f'F1 complet = {f1_base_ablation_correction:.4f}')
ax.set_xlabel('Nombre de features retirees')
ax.set_ylabel('F1 macro')
ax.set_title('Cumulative Ablation MARBERT CORRIGE - ArSarcaMoji (18 features)')
ax.legend()
ax.set_xticks(etapes)
plt.tight_layout()
plt.savefig("../outputs_dedup_correction/cumulative_ablation_final.png")
plt.show()
print("✅ Graphique sauvegarde !")
